# 02 - Data Cleaning & Feature Engineering
**Objective:** build one consistent analytical dataset per window using the *same* feature function.
- Data All -> Q1-Q4, Data Validasi -> Q2-Q5
- No rows deleted, target never used to build predictors, Customer ID never a feature.

Outputs are cleared before committing (customer-level data).

In [ ]:
import sys
sys.path.append("..")           # lets us import from ../src
import pandas as pd, numpy as np
from src.preprocessing import load_clean, TARGET, ID_COL, PROFILE_COLS
from src.feature_engineering import (build_features, WINDOW_ALL, WINDOW_VALIDASI,
    detect_constant_features, assert_same_schema, redundant_pairs, CATEGORICAL_FEATURES)
pd.set_option("display.max_columns", 60)

clean_data_all = load_clean("../data/raw/finanku_data_all.csv")
clean_data_validasi = load_clean("../data/raw/finanku_data_validasi.csv")
print(clean_data_all.shape, clean_data_validasi.shape)
print(clean_data_all.columns.tolist())

## 1. Cleaning checks
`load_clean` validates ranges/types and raises an error if anything is wrong. Nothing is dropped or imputed.

In [ ]:
print(clean_data_all.dtypes.value_counts())
print("branch_code dtype:", clean_data_all["branch_code"].dtype, "| city dtype:", clean_data_all["city"].dtype)

# Shared columns must still match after cleaning
shared = [c for c in clean_data_all.columns if c in clean_data_validasi.columns and c != TARGET]
print("Shared columns identical:", all((clean_data_all[c].values == clean_data_validasi[c].values).all() for c in shared))

## 2. Build features with the same function

In [ ]:
engineered_features_all = build_features(clean_data_all, WINDOW_ALL)             # q1..q4
engineered_features_validasi = build_features(clean_data_validasi, WINDOW_VALIDASI)  # q2..q5

assert_same_schema(engineered_features_all, engineered_features_validasi)
print("Shapes:", engineered_features_all.shape, engineered_features_validasi.shape)
print("Target in features:", TARGET in engineered_features_all.columns)
print("ID is index, not a feature:", ID_COL not in engineered_features_all.columns)
print(engineered_features_all.columns.tolist())

## 3. Manual spot-check (proves the formulas)

In [ ]:
rng = np.random.default_rng(42)
for i in rng.choice(len(clean_data_all), 5, replace=False):
    r = clean_data_all.iloc[i]
    bal = [r[f"balance_{q}"] for q in WINDOW_ALL]
    f = engineered_features_all.iloc[i]
    assert np.isclose(f["balance_mean"], np.mean(bal))
    assert np.isclose(f["balance_std"], np.std(bal, ddof=1))
    assert f["balance_change_abs"] == bal[-1] - bal[0]
    if bal[0] > 0:
        assert np.isclose(f["balance_change_pct"], (bal[-1] - bal[0]) / bal[0])
    else:
        assert np.isnan(f["balance_change_pct"]) and f["balance_start_is_zero"] == 1
    assert f["n_zero_balance_quarters"] == sum(b == 0 for b in bal)
print("Spot-checks passed")

## 4. Zero starting balances and missing values
`balance_change_pct` is undefined when the starting balance is 0, so it is NaN and flagged by `balance_start_is_zero`.

In [ ]:
for name, F in [("All (Q1-Q4)", engineered_features_all), ("Validasi (Q2-Q5)", engineered_features_validasi)]:
    print(f"{name}: zero-start customers = {int(F['balance_start_is_zero'].sum()):,} "
          f"({F['balance_start_is_zero'].mean():.1%}) | NaN in pct change = {int(F['balance_change_pct'].isna().sum()):,}")
    print("  columns with NaN:", F.columns[F.isna().any()].tolist())

## 5. Constant features and redundancy

In [ ]:
for name, F in [("All", engineered_features_all), ("Validasi", engineered_features_validasi)]:
    print(name, "constant features:", detect_constant_features(F))

print("\nHighly correlated pairs (|r| >= 0.95), Data All:")
display(redundant_pairs(engineered_features_all))

## 6. Credit-card features differ by window
The card indicator is identical Q1-Q3 and 1 in Q4/Q5, so card features mean something different in each window. Treat them with care in modeling.

In [ ]:
print("n_card_quarters, All:", engineered_features_all["n_card_quarters"].value_counts().sort_index().to_dict())
print("n_card_quarters, Validasi:", engineered_features_validasi["n_card_quarters"].value_counts().sort_index().to_dict())
print("card_start == 0: All", int((engineered_features_all["card_start"] == 0).sum()),
      "| Validasi", int((engineered_features_validasi["card_start"] == 0).sum()))

## 7. Window comparison of engineered features (aggregate)

In [ ]:
num = engineered_features_all.select_dtypes("number").columns
cmp = pd.DataFrame({"All_mean": engineered_features_all[num].mean(),
                    "Validasi_mean": engineered_features_validasi[num].mean()})
cmp["diff"] = cmp["Validasi_mean"] - cmp["All_mean"]
display(cmp.round(4))

## 8. Save processed data locally (git-ignored, never commit)

In [ ]:
import os
os.makedirs("../data/processed", exist_ok=True)
clean_data_all.to_csv("../data/processed/clean_data_all.csv", index=False)
clean_data_validasi.to_csv("../data/processed/clean_data_validasi.csv", index=False)
engineered_features_all.to_csv("../data/processed/engineered_features_all.csv")
engineered_features_validasi.to_csv("../data/processed/engineered_features_validasi.csv")
print("Saved 4 files in data/processed/")